# CplusyAIT GPTQ Ternary Quantization
Run on **Colab GPU runtime** (T4+). Do NOT run locally.

In [1]:
!pip install -q torch transformers datasets psutil numpy

In [2]:
import torch, psutil
ram = psutil.virtual_memory().total / (1024**3)
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f'RAM: {ram:.1f} GB | GPU: {gpu}')
assert ram > 10, 'Use Colab runtime, not local!'

RAM: 12.7 GB | GPU: Tesla T4


In [9]:
import os
if os.path.isdir('CplusyAIT'):
    !cd CplusyAIT && git pull
else:
    !git clone https://github.com/OCDDEVS/CplusyAIT.git
!ls CplusyAIT/scripts/

Cloning into 'CplusyAIT'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 270 (delta 130), reused 223 (delta 89), pack-reused 0 (from 0)
Receiving objects: 100% (270/270), 593.91 KiB | 3.41 MiB/s, done.
Resolving deltas: 100% (130/130), done.
download_model.py  quantize_gptq.py


In [10]:
# Quantize TinyLlama 1.1B to ternary
!python CplusyAIT/scripts/quantize_gptq.py \
  --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --output CplusyAIT/models/TinyLlama-1.1B-GPTQ-ternary \
  --mem-limit 11

Loading model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Memory limit: 11.0 GB (will abort if exceeded)
Current RAM usage: 0.8 GB
config.json: 100% 608/608 [00:00<00:00, 1.44MB/s]
tokenizer_config.json: 1.29kB [00:00, 3.05MB/s]
tokenizer.json: 1.84MB [00:00, 59.1MB/s]
tokenizer.model: 100% 500k/500k [00:00<00:00, 595kB/s] 
special_tokens_map.json: 100% 551/551 [00:00<00:00, 3.83MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 2.20G/2.20G [00:22<00:00, 96.4MB/s]
Loading weights: 100% 201/201 [00:02<00:00, 96.07it/s, Materializing param=model.norm.weight]                               
generation_config.json: 100% 124/124 [00:00<00:00, 818kB/s]
Model loaded. RAM usage: 5.1 GB
Using weight-statistics Hessian (no calibration data needed)
Model: 22 layers, hidden=2048, vocab=32000
Heads: 32 Q, 4 KV, intermediate=5632

--- Quantizing layer by layer ---
  [1/201] [FP32] model.embed_tokens.weight [32000, 2048] -> 262144000 bytes (0.7s)
  [2/201] [GPTQ] model.layers.0.se

In [11]:
# Check output
import os, json
out = 'CplusyAIT/models/TinyLlama-1.1B-GPTQ-ternary'
manifest = json.load(open(os.path.join(out, 'manifest.json')))
wb = os.path.getsize(os.path.join(out, 'weights.bin')) / (1024**2)
print(f'Tensors: {len(manifest["tensors"])}')
print(f'Weights: {wb:.1f} MB')

Tensors: 356
Weights: 498.6 MB


## Run inference on Colab
Install Rust toolchain, compile the project, and run the ternary inference engine.

In [ ]:
# Install Rust (takes ~30s)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version

In [ ]:
# Build the smoke test binary (~2-3 min first time)
!cd CplusyAIT && cargo build --release --bin smoke_test 2>&1 | tail -5

In [ ]:
# Build the infer binary
!cd CplusyAIT && cargo build --release --bin infer 2>&1 | tail -5

In [ ]:
# Run inference with the GPTQ-quantized model
!cd CplusyAIT && ./target/release/infer \
  --model models/TinyLlama-1.1B-GPTQ-ternary \
  --prompt "The meaning of life is" \
  --max-tokens 50 \
  --temperature 0.7

In [ ]:
# Download the quantized model to your machine
from google.colab import files
!cd CplusyAIT/models && zip -r /tmp/TinyLlama-ternary.zip TinyLlama-1.1B-GPTQ-ternary/
files.download('/tmp/TinyLlama-ternary.zip')